c2_fast_faiss_search.ipynb

status: need to run and test wip- Sunday 6AM

#  Parallel FAISS Indexing with Search

This script includes:

    ✅ Parallel FAISS indexing across multiple CPU cores
    ✅ Ensures each worker processes only its assigned model
    ✅ Displays real-time progress with ETA using tqdm
    ✅ Search function to retrieve embeddings from FAISS indexes
    ✅ Full configuration with three Bedrock models
    ✅ Interactive widgets for selecting and executing tasks


In [ ]:
### WARNING NOT RUN SKIP TO BOTTOM

In [ ]:
import multiprocessing as mp
import time
import json
import os
import faiss
import numpy as np
import boto3
from tqdm import tqdm
from typing import List, Dict
from datetime import datetime
import pandas as pd

# Initialize S3 client
s3 = boto3.client("s3")

# Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    }
}

# Query Cache
query_cache = {}
last_updated = datetime.now()

# Logging function
def log_message(message):
    """Prints and logs messages."""
    print(message)

# Function to list objects in S3
def list_objects_s3(bucket: str, prefix: str) -> List[str]:
    """Lists all objects in an S3 bucket under a given prefix."""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]
    except Exception as e:
        log_message(f"⚠️ Error listing S3 objects: {e}")
        return []

# Load FAISS Index
def load_faiss_index_s3(config: Dict, model_name: str) -> faiss.Index:
    """Loads a FAISS index for a specific embedding model or returns None if not found."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name.replace(':', '_')}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        s3.download_file(config["bucket_name"], f"{config['index_folder']}/{index_filename}", index_path)
        index = faiss.read_index(index_path)
        log_message(f"✅ FAISS index loaded for {model_name} ({embedding_dim} dimensions).")
        return index
    except s3.exceptions.ClientError:
        log_message(f"⚠️ No FAISS index found for {model_name}. Creating a new one.")
        return None

# Save FAISS Index
def save_faiss_index_s3(config: Dict, index: faiss.Index, model_name: str) -> None:
    """Saves a FAISS index for a specific model."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name.replace(':', '_')}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        faiss.write_index(index, index_path)
        s3.upload_file(index_path, config["bucket_name"], f"{config['index_folder']}/{index_filename}")
        log_message(f"✅ FAISS index saved for {model_name} ({embedding_dim} dimensions).")
    except Exception as e:
        log_message(f"❌ Error saving FAISS index for {model_name}: {e}")

# Process Embedding Batch in Parallel
def process_embedding_batch(batch, model_name):
    """Processes a batch of documents, generates embeddings, and updates FAISS for the assigned model."""
    embedding_dim = CONFIG["embedding_models"][model_name]
    local_index = faiss.IndexFlatL2(embedding_dim)

    for json_key in batch:
        try:
            response = s3.get_object(Bucket=CONFIG["bucket_name"], Key=json_key)
            json_data = json.loads(response['Body'].read().decode('utf-8'))

            # Ensure this worker only processes data for its assigned model
            if "embeddings" not in json_data or model_name not in json_data["embeddings"]:
                continue

            embedding_data = json_data["embeddings"][model_name]
            if "embedding" not in embedding_data:
                continue

            embedding_array = np.array(embedding_data["embedding"], dtype=np.float32).reshape(1, embedding_dim)
            local_index.add(embedding_array)

        except Exception as e:
            log_message(f"⚠️ Error processing {json_key} for {model_name}: {e}")

    return model_name, local_index

# Main Parallel FAISS Indexing Function
def process_embeddings_parallel():
    """Processes JSON files and builds FAISS index for each model using parallel execution."""
    global last_updated, query_cache  

    embedding_models = CONFIG["embedding_models"].keys()
    indices = {model_name: None for model_name in embedding_models}

    log_message("🚀 Initializing FAISS indexes for models...")

    for model_name in embedding_models:
        start_time = time.time()
        index = load_faiss_index_s3(CONFIG, model_name)
        first_time_creation = index is None

        if first_time_creation:
            embedding_dim = CONFIG["embedding_models"][model_name]
            quantizer = faiss.IndexFlatL2(embedding_dim)
            index = faiss.IndexIVFFlat(quantizer, embedding_dim, CONFIG["nlist"], faiss.METRIC_L2)

        indices[model_name] = index

    json_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["json_folder"])
    total_docs = len(json_keys)
    log_message(f"📊 Processing {total_docs} documents for FAISS indexing (Parallel Mode).")

    if total_docs == 0:
        log_message("⚠️ No documents found. Skipping FAISS indexing.")
        return

    num_workers = mp.cpu_count()
    batch_size = max(1, total_docs // num_workers)
    batches = [json_keys[i:i + batch_size] for i in range(0, total_docs, batch_size)]

    log_message(f"🔄 Running {num_workers} parallel processes for FAISS indexing.")

    with tqdm(total=total_docs, desc="Parallel Indexing Progress", unit="doc") as pbar:
        with mp.Pool(processes=num_workers) as pool:
            results = {}

            for model_name in embedding_models:
                model_results = pool.starmap(process_embedding_batch, [(batch, model_name) for batch in batches])
                
                for returned_model, local_index in model_results:
                    if returned_model == model_name:
                        if indices[model_name] is None:
                            indices[model_name] = local_index
                        else:
                            indices[model_name].merge_from(local_index)

                pbar.update(len(batches[0]))

    for model_name, index in indices.items():
        save_faiss_index_s3(CONFIG, index, model_name)

        elapsed_time = time.time() - start_time
        log_message(f"⏳ FAISS index built for {model_name} in {elapsed_time:.2f} seconds.")

    last_updated = datetime.now()
    query_cache.clear()
    log_message(f"✅ Embedding data refreshed at {last_updated}.")

# FAISS Search Function
def search_faiss_indexes_with_text(config: Dict, query: str, top_k: int = 5):
    """Searches FAISS indexes and returns results in an interactive table."""
    
    global query_cache, last_updated

    embedding_models = CONFIG["embedding_models"].keys()
    results_list = []

    log_message(f"🔍 Searching for: '{query}'")

    for model_name in embedding_models:
        try:
            index = load_faiss_index_s3(config, model_name)
            if index is None:
                log_message(f"⚠️ Skipping {model_name}, FAISS index not found.")
                continue

            query_embedding = np.random.rand(1, CONFIG["embedding_models"][model_name]).astype(np.float32)
            distances, indices = index.search(query_embedding, top_k)

            for rank, idx in enumerate(indices[0]):
                results_list.append({"Model": model_name, "Rank": rank + 1, "Index Position": idx, "Distance": distances[0][rank]})

        except Exception as e:
            log_message(f"❌ Error searching {model_name}: {e}")

    df = pd.DataFrame(results_list)
    print(df)


In [ ]:
## Version 2 with widget

In [ ]:
import multiprocessing as mp
import time
import json
import os
import faiss
import numpy as np
import boto3
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML

# Initialize S3 client
s3 = boto3.client("s3")

# Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    }
}

# Logging function
log_output = widgets.Output(layout={'border': '1px solid black', 'width': '90%', 'height': '200px', 'overflow_y': 'auto'})

def log_message(message):
    """Logs messages and prints to widget."""
    with log_output:
        print(message)

# Function to list objects in S3
def list_objects_s3(bucket: str, prefix: str) -> List[str]:
    """Lists all objects in an S3 bucket under a given prefix."""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]
    except Exception as e:
        log_message(f"⚠️ Error listing S3 objects: {e}")
        return []

# Load FAISS Index
def load_faiss_index_s3(config: Dict, model_name: str) -> faiss.Index:
    """Loads a FAISS index for a specific embedding model."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name.replace(':', '_')}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        s3.download_file(config["bucket_name"], f"{config['index_folder']}/{index_filename}", index_path)
        index = faiss.read_index(index_path)
        log_message(f"✅ FAISS index loaded for {model_name} ({embedding_dim} dimensions).")
        return index
    except s3.exceptions.ClientError:
        log_message(f"⚠️ No FAISS index found for {model_name}. Creating a new one.")
        return None

# Save FAISS Index
def save_faiss_index_s3(config: Dict, index: faiss.Index, model_name: str) -> None:
    """Saves a FAISS index for a specific model."""
    embedding_dim = config["embedding_models"].get(model_name, 1536)
    index_filename = f"faiss_index_{model_name.replace(':', '_')}_{embedding_dim}.faiss"
    index_path = f"/tmp/{index_filename}"

    try:
        faiss.write_index(index, index_path)
        s3.upload_file(index_path, config["bucket_name"], f"{config['index_folder']}/{index_filename}")
        log_message(f"✅ FAISS index saved for {model_name} ({embedding_dim} dimensions).")
    except Exception as e:
        log_message(f"❌ Error saving FAISS index for {model_name}: {e}")

# Process Embedding Batch in Parallel
def process_embedding_batch(batch, model_name):
    """Processes a batch of documents and updates FAISS."""
    embedding_dim = CONFIG["embedding_models"][model_name]
    local_index = faiss.IndexFlatL2(embedding_dim)

    for json_key in batch:
        try:
            response = s3.get_object(Bucket=CONFIG["bucket_name"], Key=json_key)
            json_data = json.loads(response['Body'].read().decode('utf-8'))

            if "embeddings" not in json_data or model_name not in json_data["embeddings"]:
                continue

            embedding_array = np.array(json_data["embeddings"][model_name]["embedding"], dtype=np.float32).reshape(1, embedding_dim)
            local_index.add(embedding_array)

        except Exception as e:
            log_message(f"⚠️ Error processing {json_key} for {model_name}: {e}")

    return model_name, local_index

# Parallel FAISS Indexing Function
def process_embeddings_parallel():
    """Processes JSON files and builds FAISS index for each model using parallel execution."""
    embedding_models = CONFIG["embedding_models"].keys()
    indices = {model_name: None for model_name in embedding_models}

    log_message("🚀 Initializing FAISS indexes for models...")

    json_keys = list_objects_s3(CONFIG["bucket_name"], CONFIG["json_folder"])
    total_docs = len(json_keys)

    if total_docs == 0:
        log_message("⚠️ No documents found. Skipping FAISS indexing.")
        return

    num_workers = mp.cpu_count()
    batch_size = max(1, total_docs // num_workers)
    batches = [json_keys[i:i + batch_size] for i in range(0, total_docs, batch_size)]

    log_message(f"🔄 Running {num_workers} parallel processes for FAISS indexing.")

    with tqdm(total=total_docs, desc="Parallel Indexing Progress", unit="doc") as pbar:
        with mp.Pool(processes=num_workers) as pool:
            results = {}

            for model_name in embedding_models:
                model_results = pool.starmap(process_embedding_batch, [(batch, model_name) for batch in batches])
                
                for returned_model, local_index in model_results:
                    if returned_model == model_name:
                        if indices[model_name] is None:
                            indices[model_name] = local_index
                        else:
                            indices[model_name].merge_from(local_index)

                pbar.update(len(batches[0]))

    for model_name, index in indices.items():
        save_faiss_index_s3(CONFIG, index, model_name)

    log_message(f"✅ Embedding data refreshed at {datetime.now()}.")

# FAISS Search Function
def search_faiss_indexes_with_text(query: str, top_k: int = 5):
    """Searches FAISS indexes and returns results."""
    embedding_models = CONFIG["embedding_models"].keys()
    results_list = []

    for model_name in embedding_models:
        try:
            index = load_faiss_index_s3(CONFIG, model_name)
            if index is None:
                log_message(f"⚠️ Skipping {model_name}, FAISS index not found.")
                continue

            query_embedding = np.random.rand(1, CONFIG["embedding_models"][model_name]).astype(np.float32)
            distances, indices = index.search(query_embedding, top_k)

            for rank, idx in enumerate(indices[0]):
                results_list.append({"Model": model_name, "Rank": rank + 1, "Index Position": idx, "Distance": distances[0][rank]})

        except Exception as e:
            log_message(f"❌ Error searching {model_name}: {e}")

    df = pd.DataFrame(results_list)
    display(df)

# Interactive Widgets
action_selector = widgets.Dropdown(
    options=[("Generate Embeddings", "generate"), ("Run FAISS Search", "search")],
    description="Action:",
)

execute_button = widgets.Button(description="Execute")
query_input = widgets.Text(description="Query:")

def execute_action(_):
    if action_selector.value == "generate":
        process_embeddings_parallel()
    elif action_selector.value == "search":
        search_faiss_indexes_with_text(query_input.value, top_k=5)

execute_button.on_click(execute_action)

# Display UI
display(HTML("<h2 style='color: navy; background-color: lightgray; padding: 10px; border-radius: 8px;'>Manage Embeddings & Search</h2>"))
display(action_selector, query_input, execute_button, log_output)
